In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import time

sys.path.append('../src')

In [3]:
from llm_val.utils.api import GigaChatAPI

# GigaChatAPI

`GigaChatAPI` - это класс, реализующий функционал API GigaChat'а. Подробнее о **возможностях(функционале) класса** можно узнать в соответствующем разделе данной тетрадки.

На данный момент к API GigaChat'а можно подключиться из трех 'контуров': *MLSpace*, *Sigma* и *DataLab* (далее по тексту слово API опускается и при упоминании того или иного контура подразумевается API в этом контуре).

Особенности API'шек:

1. У *Sigma* и *MLSpace* различные ссылки для получения токена, у *DataLab* токен получают из переменных окружения.
2. В *DataLab* нельзя делать больше одного запроса в 5 секунд.
3. Ссылка к API *Datalab* отличается от ссылки в *Sigma* и *MLSpace*.

In [4]:
RQUID = 'INSERT_YOUR_CLIENT_ID_HERE'
AUTH_DATA = 'INSERT_YOUR_CLIENT_SECRET_HERE'

In [5]:
# OAUTH_URL = 'https://ngw.devices.sberbank.ru:9443/api/v2/oauth' #MLSpace
# API_URL = 'https://gigachat.devices.sberbank.ru/api/v1' #MLSpace
# TOKEN = None #MLSpace

In [6]:
OAUTH_URL = 'https://sm-auth-sd.prom-88-89-apps.ocp-geo.ocp.sigma.sbrf.ru/api/v2/oauth' #Sigma
API_URL = 'https://gigachat.devices.sberbank.ru/api/v1' #Sigma
TOKEN = None #Sigma

In [7]:
# OAUTH_URL = None #datalab
# API_URL = 'http://liveaccess/v1/gc' #datalab
# TOKEN = os.environ.get('JPY_API_TOKEN') #datalab

Конструктор класса принимает в себя **6 опциональных параметров**.

```python
class GigaChatAPI:
    def __init__(self,
                 contour: tp.Optional[str]=None,
                 token: tp.Optional[str]=None,
                 auth_data: tp.Optional[str]=None,
                 api_url: tp.Optional[str]=None,
                 oauth_url: tp.Optional[str]=None,
                 rquid: tp.Optional[str]=None):
        '''
        contour: Название контура. Принимает одно из трех значениий ['sigma', 'mlspace', 'datalab']
        token: Токен для авторизации. Передается, если уже получен
        auth_data: Авторизационные данные: Client id и Client Secret, закодированные в Base64
        api_url: URL до API GigaChat 
        oauth_url: URL до сервиса авторизации GigaChat
        rquid: Уникальный идентификатор запроса. Соответствует формату uuid4.
                Параметр для журналирования входящих вызовов и разбора инцидентов.
        '''
```
Можно передать или `contour`, или `api_url` и `oauth_url`; если переданы и `*_url`, и `contour`, то в приоритетете `*_url`.

Для *Sigma* и *MLSpace* есть свои обязательные параметры, для *Datalab* - свои:

- *Sigma* и *MLSpace*:
    - `contour` или `*_url`;
    - `auth_data`.
- *DataLab*:
    - `contour` или `*_url`;
    - `token`.

In [8]:
api = GigaChatAPI(contour=None,
                  token=TOKEN,
                  auth_data=AUTH_DATA,
                  api_url=API_URL,
                  oauth_url=OAUTH_URL,
                  rquid=RQUID)

# Возможности GigaChatAPI

## Проверка "свежести" токена и  его апдейт

In [9]:
print(f'Текущее время: {int(time.time() * 1000)}')
print(f'Время истечения токена: {api.token_expires_time}')
print(f'Истек ли токен? {api.is_token_expires()}')

Текущее время: 1717398315355
Время истечения токена: 1717400113780
Истек ли токен? False


In [10]:
api.update_token(AUTH_DATA)

In [11]:
print(f'Текущее время: {int(time.time()  * 1000)}')
print(f'Время истечения токена: {api.token_expires_time}')
print(f'Истек ли токен? {api.is_token_expires()}')

Текущее время: 1717398317406
Время истечения токена: 1717400116673
Истек ли токен? False


## Получить список моделей

In [12]:
api.get_list_models()

[{'id': 'GigaChat', 'object': 'model', 'owned_by': 'salutedevices'},
 {'id': 'GigaChat-Plus', 'object': 'model', 'owned_by': 'salutedevices'},
 {'id': 'GigaChat-Plus-preview',
  'object': 'model',
  'owned_by': 'salutedevices'},
 {'id': 'GigaChat-Pro', 'object': 'model', 'owned_by': 'salutedevices'},
 {'id': 'GigaChat-Pro-preview',
  'object': 'model',
  'owned_by': 'salutedevices'},
 {'id': 'GigaChat-preview', 'object': 'model', 'owned_by': 'salutedevices'}]

## Получить ответ модели на сообщения

In [13]:
messages = [{'role': 'system', 
             'content': ''},
            {'role': 'user', 
             'content': 'Тестовое сообщение. Напиши "капибара", если меня понимаешь.'}]

In [14]:
api.get_answer_gigachat(messages)

[{'message': {'content': 'Капибара.', 'role': 'assistant'},
  'index': 0,
  'finish_reason': 'stop'}]

## Создать эмбеддинг

In [15]:
corpus = ['Привет. Я маленькая капибара', 
          'Перемен - требуют наши сердца', 
          'Расстояние, версты, мили. Нас расставили, рассадили',
          'Грохочет гром. Сверкает молния в ночи.']

In [ ]:
api.get_embedding(corpus=corpus)

## Подсчитать количество токенов

In [17]:
corpus = ['Привет. Я маленькая капибара', 
          'Перемен - требуют наши сердца', 
          'Расстояние, версты, мили. Нас расставили, рассадили',
          'Грохочет гром. Сверкает молния в ночи.']

In [18]:
api.get_token_count(corpus=corpus)

[{'object': 'tokens', 'tokens': 9, 'characters': 28},
 {'object': 'tokens', 'tokens': 6, 'characters': 29},
 {'object': 'tokens', 'tokens': 17, 'characters': 51},
 {'object': 'tokens', 'tokens': 13, 'characters': 38}]